# DermaXplain - Phase 5A: Pseudo-Concept Map Generation

This notebook generates **clinically inspired pseudo-concept maps** for later XAI evaluation.

The maps are not clinical ground-truth annotations. They are deterministic, spatial proxies derived from the lesion mask, and optionally from the RGB image.

## Main pseudo-concepts

1. **Asymmetry evidence map (A)**  
   Regions of the lesion that do not match their reflected counterpart across the main lesion axes.

2. **Border irregularity evidence map (B)**  
   Boundary regions where the original lesion border deviates from a smoothed reference border.

3. **Colour heterogeneity proxy (C, optional)**  
   Lesion pixels whose colour differs from the typical lesion colour. This is included as a weak heuristic only.

## Explicitly excluded

Diameter is not implemented as a clinical pseudo-concept because HAM10000 / ISIC masks do not provide reliable physical scale in mm per pixel.

All processing is performed in **224 × 224 model-input space** so that pseudo-concept maps align directly with future ResNet-50 inputs and Grad-CAM / LIME / SHAP outputs.


## 1. Imports and project paths

This follows the same conventions as `01_data_preprocessing.ipynb`.


In [ ]:
from pathlib import Path
import random
import warnings

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

from scipy.ndimage import (
    binary_erosion,
    binary_dilation,
    binary_fill_holes,
    distance_transform_edt,
    gaussian_filter,
    gaussian_filter1d,
)

from skimage.measure import find_contours

try:
    from skimage.color import rgb2lab
    SKIMAGE_AVAILABLE = True
except Exception:
    SKIMAGE_AVAILABLE = False
    warnings.warn("scikit-image not available. Colour proxy will fall back to RGB space.")

pd.set_option("disp" \
"lay.max_columns", None)
pd.set_option("display.max_rows", 30)

RANDOM_STATE = 42
TARGET_SIZE = (224, 224) # image size for model input 


In [ ]:
# Adjust if notebook location changes
ROOT = Path("..")

DATA_DIR = ROOT / "data"
MANIFEST_DIR = DATA_DIR / "preprocessed_manifests"

ISIC_DIR = DATA_DIR / "isic2018"
ISIC_IMG_DIR = ISIC_DIR / "images"
ISIC_MASK_DIR = ISIC_DIR / "masks"

HAM_DIR = DATA_DIR / "ham10000"
HAM_IMG_DIR = HAM_DIR / "images"
HAM_MASK_DIR = HAM_DIR / "masks"

# Prefer preprocessed manifests if they already exist, otherwise fall back to inventory manifests.
ISIC_PREPROCESSED = MANIFEST_DIR / "isic2018_preprocessed.csv"
HAM_PREPROCESSED = MANIFEST_DIR / "ham10000_preprocessed.csv"
ISIC_INVENTORY = MANIFEST_DIR / "isic2018_inventory.csv"
HAM_INVENTORY = MANIFEST_DIR / "ham10000_inventory.csv"

ISIC_MANIFEST = ISIC_PREPROCESSED if ISIC_PREPROCESSED.exists() else ISIC_INVENTORY
HAM_MANIFEST = HAM_PREPROCESSED if HAM_PREPROCESSED.exists() else HAM_INVENTORY

assert ISIC_MANIFEST.exists(), f"Missing manifest: {ISIC_MANIFEST.resolve()}"
assert HAM_MANIFEST.exists(), f"Missing manifest: {HAM_MANIFEST.resolve()}"

PSEUDO_DIR = DATA_DIR / "pseudo_concepts"
PSEUDO_NPZ_DIR = PSEUDO_DIR / "npz"
PSEUDO_PNG_DIR = PSEUDO_DIR / "png_preview"
PSEUDO_MANIFEST_DIR = PSEUDO_DIR / "manifests"

for d in [PSEUDO_NPZ_DIR, PSEUDO_PNG_DIR, PSEUDO_MANIFEST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Using manifests:")
print("-", ISIC_MANIFEST.resolve())
print("-", HAM_MANIFEST.resolve())
print("\nOutput directory:")
print("-", PSEUDO_DIR.resolve())


## 2. Load manifests and select usable rows

Rows must have both image and mask available. If the manifest contains `same_size`, this notebook also requires it.


In [ ]:
isic_df = pd.read_csv(ISIC_MANIFEST)
ham_df = pd.read_csv(HAM_MANIFEST)

print("ISIC shape:", isic_df.shape)
print("HAM shape :", ham_df.shape)

def select_usable_rows(df: pd.DataFrame) -> pd.DataFrame:
    required = df["image_exists"].astype(bool) & df["mask_exists"].astype(bool)
    if "same_size" in df.columns:
        required = required & df["same_size"].astype(bool)
    return df.loc[required].copy().reset_index(drop=True)

isic_core = select_usable_rows(isic_df)
ham_core = select_usable_rows(ham_df)

print("ISIC usable rows:", len(isic_core))
print("HAM usable rows :", len(ham_core))

display(ham_core.head())


## 3. Reusable image and mask helpers

Images are resized using bilinear interpolation. Masks are resized using nearest-neighbor interpolation to preserve binary labels.


In [ ]:
def load_rgb(path: Path) -> np.ndarray:
    return np.array(Image.open(path).convert("RGB"), dtype=np.uint8)


def load_mask_gray(path: Path) -> np.ndarray:
    return np.array(Image.open(path).convert("L"))


def to_mask_bin(mask_gray: np.ndarray) -> np.ndarray:
    return (mask_gray > 0).astype(np.uint8)


def resize_image_rgb(img_rgb: np.ndarray, size=TARGET_SIZE) -> np.ndarray:
    return np.array(Image.fromarray(img_rgb).resize(size, Image.Resampling.BILINEAR))


def resize_mask_bin(mask_bin: np.ndarray, size=TARGET_SIZE) -> np.ndarray:
    mask_img = Image.fromarray((mask_bin * 255).astype(np.uint8))
    mask_resized = mask_img.resize(size, Image.Resampling.NEAREST)
    return (np.array(mask_resized) > 0).astype(np.uint8)


def overlay_mask(img_rgb: np.ndarray, mask_bin: np.ndarray, alpha: float = 0.35) -> np.ndarray:
    out = img_rgb.copy().astype(np.float32)
    color = np.zeros_like(out)
    color[..., 0] = 255
    lesion = mask_bin.astype(bool)
    out[lesion] = (1 - alpha) * out[lesion] + alpha * color[lesion]
    return out.astype(np.uint8)


def normalize_01(x: np.ndarray, mask: np.ndarray | None = None, eps: float = 1e-8) -> np.ndarray:
    y = x.astype(np.float32).copy()
    region = mask.astype(bool) if mask is not None else np.ones_like(y, dtype=bool)
    if not np.any(region):
        return np.zeros_like(y, dtype=np.float32)
    vals = y[region]
    lo, hi = float(vals.min()), float(vals.max())
    if hi - lo < eps:
        return np.zeros_like(y, dtype=np.float32)
    y = (y - lo) / (hi - lo + eps)
    y[~region] = 0.0
    return y.astype(np.float32)


## 4. Core geometric helpers

The centroid and PCA axes are used for the asymmetry map. The PCA axis gives the dominant orientation of the lesion in resized 224 × 224 space.


In [ ]:
def lesion_coordinates(mask_bin: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    ys, xs = np.where(mask_bin > 0)
    return ys.astype(np.float32), xs.astype(np.float32)


def compute_centroid(mask_bin: np.ndarray) -> tuple[float, float]:
    ys, xs = lesion_coordinates(mask_bin)
    if len(xs) == 0:
        return np.nan, np.nan
    return float(xs.mean()), float(ys.mean())


def compute_pca_axes(mask_bin: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return centroid, major axis unit vector, and minor axis unit vector.

    Coordinates are represented as [x, y].
    """
    ys, xs = lesion_coordinates(mask_bin)
    if len(xs) < 3:
        centroid = np.array([np.nan, np.nan], dtype=np.float32)
        return centroid, np.array([1.0, 0.0]), np.array([0.0, 1.0])

    coords = np.column_stack([xs, ys]).astype(np.float32)
    centroid = coords.mean(axis=0)
    centered = coords - centroid
    cov = np.cov(centered, rowvar=False)
    eigvals, eigvecs = np.linalg.eigh(cov)
    order = np.argsort(eigvals)[::-1]
    major = eigvecs[:, order[0]].astype(np.float32)
    major = major / (np.linalg.norm(major) + 1e-8)
    minor = np.array([-major[1], major[0]], dtype=np.float32)
    return centroid.astype(np.float32), major, minor


def reflect_mask_across_axis(mask_bin: np.ndarray, centroid: np.ndarray, axis: np.ndarray) -> np.ndarray:
    """Reflect foreground mask pixels across a line passing through centroid.

    The axis is a 2D unit vector in [x, y] coordinates.
    """
    h, w = mask_bin.shape
    ys, xs = np.where(mask_bin > 0)
    if len(xs) == 0:
        return np.zeros_like(mask_bin, dtype=np.uint8)

    coords = np.column_stack([xs, ys]).astype(np.float32)
    centered = coords - centroid[None, :]

    # Reflection matrix across line with unit direction u: R = 2 u u^T - I
    u = axis.astype(np.float32)
    u = u / (np.linalg.norm(u) + 1e-8)
    reflection_matrix = 2.0 * np.outer(u, u) - np.eye(2, dtype=np.float32)
    reflected = centered @ reflection_matrix.T + centroid[None, :]

    xr = np.rint(reflected[:, 0]).astype(int)
    yr = np.rint(reflected[:, 1]).astype(int)
    valid = (xr >= 0) & (xr < w) & (yr >= 0) & (yr < h)

    reflected_mask = np.zeros_like(mask_bin, dtype=np.uint8)
    reflected_mask[yr[valid], xr[valid]] = 1
    reflected_mask = binary_fill_holes(reflected_mask).astype(np.uint8)
    return reflected_mask


## 5. Pseudo-concept A: asymmetry evidence map

This map estimates where lesion pixels do not have a corresponding reflected lesion pixel.

We compute asymmetry across both the PCA major axis and the orthogonal minor axis, then combine them.


In [ ]:
def asymmetry_maps(mask_bin: np.ndarray) -> dict:
    """Generate asymmetry evidence maps from a binary lesion mask.

    Returns:
    - asym_major: mismatch inside original lesion after reflection across major axis
    - asym_minor: mismatch inside original lesion after reflection across minor axis
    - asym_combined: maximum of both maps
    - reflected_major, reflected_minor: reflected masks for debugging
    """
    mask = mask_bin.astype(np.uint8)
    centroid, major, minor = compute_pca_axes(mask)

    if np.isnan(centroid).any():
        z = np.zeros_like(mask, dtype=np.float32)
        return {
            "asym_major": z,
            "asym_minor": z,
            "asym_combined": z,
            "reflected_major": mask.copy(),
            "reflected_minor": mask.copy(),
            "centroid": centroid,
            "major_axis": major,
            "minor_axis": minor,
        }

    reflected_major = reflect_mask_across_axis(mask, centroid, major)
    reflected_minor = reflect_mask_across_axis(mask, centroid, minor)

    # Evidence is restricted to the original lesion, because later saliency evaluation focuses on lesion pixels.
    asym_major = ((mask == 1) & (reflected_major == 0)).astype(np.float32)
    asym_minor = ((mask == 1) & (reflected_minor == 0)).astype(np.float32)
    asym_combined = np.maximum(asym_major, asym_minor).astype(np.float32)

    return {
        "asym_major": asym_major,
        "asym_minor": asym_minor,
        "asym_combined": asym_combined,
        "reflected_major": reflected_major.astype(np.uint8),
        "reflected_minor": reflected_minor.astype(np.uint8),
        "centroid": centroid,
        "major_axis": major,
        "minor_axis": minor,
    }


## 6. Pseudo-concept B: border irregularity evidence map

This map estimates where the lesion border deviates from a smoothed reference contour.

Implementation idea:



In [ ]:


def mask_boundary(mask_bin: np.ndarray) -> np.ndarray:
    """Return one-pixel lesion boundary."""
    mask = mask_bin.astype(bool)
    eroded = binary_erosion(mask)
    boundary = mask & ~eroded
    return boundary.astype(np.uint8)


def inner_border_band(mask_bin: np.ndarray, width: int = 8) -> np.ndarray:
    """Create an inner band along the lesion border."""
    mask = mask_bin.astype(bool)
    eroded = mask.copy()

    for _ in range(width):
        eroded = binary_erosion(eroded)

    band = mask & ~eroded
    return band.astype(np.uint8)


def get_largest_contour(mask_bin: np.ndarray) -> np.ndarray:
    """Extract the largest contour from a binary lesion mask.

    Returns contour as array of shape (N, 2), with columns [y, x].
    """
    contours = find_contours(mask_bin.astype(float), level=0.5)

    if len(contours) == 0:
        return np.empty((0, 2), dtype=np.float32)

    contour = max(contours, key=len)
    return contour.astype(np.float32)


def radial_border_irregularity_map(
    mask_bin: np.ndarray,
    border_width: int = 8,
    smooth_sigma: float = 5.0,
) -> dict:
    """Generate border irregularity map using radial distance deviation."""

    mask = mask_bin.astype(bool)

    if mask.sum() == 0:
        empty = np.zeros_like(mask_bin, dtype=np.float32)
        return {
            "border_irregularity": empty,
            "boundary": empty,
            "border_band": empty,
            "radial_deviation": np.array([], dtype=np.float32),
            "radial_distance": np.array([], dtype=np.float32),
            "radial_smooth": np.array([], dtype=np.float32),
        }

    centroid, _, _ = compute_pca_axes(mask_bin)
    cx, cy = centroid

    contour = get_largest_contour(mask_bin)

    if len(contour) < 10:
        empty = np.zeros_like(mask_bin, dtype=np.float32)
        return {
            "border_irregularity": empty,
            "boundary": mask_boundary(mask_bin).astype(np.float32),
            "border_band": inner_border_band(mask_bin, border_width).astype(np.float32),
            "radial_deviation": np.array([], dtype=np.float32),
            "radial_distance": np.array([], dtype=np.float32),
            "radial_smooth": np.array([], dtype=np.float32),
        }

    ys = contour[:, 0]
    xs = contour[:, 1]

    radial_distance = np.sqrt((xs - cx) ** 2 + (ys - cy) ** 2)

    radial_smooth = gaussian_filter1d(
        radial_distance,
        sigma=smooth_sigma,
        mode="wrap",
    )

    radial_deviation = np.abs(radial_distance - radial_smooth)

    # Normalise radial deviation before projecting to image space
    if radial_deviation.max() > 0:
        radial_deviation_norm = radial_deviation / radial_deviation.max()
    else:
        radial_deviation_norm = radial_deviation

    irregularity_contour = np.zeros_like(mask_bin, dtype=np.float32)

    xi = np.rint(xs).astype(int)
    yi = np.rint(ys).astype(int)

    h, w = mask_bin.shape
    valid = (xi >= 0) & (xi < w) & (yi >= 0) & (yi < h)

    irregularity_contour[yi[valid], xi[valid]] = radial_deviation_norm[valid]

    band = inner_border_band(mask_bin, width=border_width).astype(bool)

    # Smooth lightly for visual continuity
    irregularity_map = gaussian_filter(irregularity_contour, sigma=1.2)
    irregularity_map = irregularity_map * band

    # Percentile normalisation improves visibility
    positive_values = irregularity_map[irregularity_map > 0]

    if len(positive_values) > 0:
        p95 = np.percentile(positive_values, 95)
        if p95 > 0:
            irregularity_map = np.clip(irregularity_map / p95, 0, 1)

    return {
        "border_irregularity": irregularity_map.astype(np.float32),
        "boundary": mask_boundary(mask_bin).astype(np.float32),
        "border_band": band.astype(np.float32),
        "radial_deviation": radial_deviation_norm.astype(np.float32),
        "radial_distance": radial_distance.astype(np.float32),
        "radial_smooth": radial_smooth.astype(np.float32),
    }

## 7. Optional pseudo-concept C: colour heterogeneity proxy

This is weaker than A and B because it is not based on expert colour annotations. It is included only as a heuristic.

The map highlights lesion pixels whose colour differs from the typical lesion colour. LAB space is preferred if `scikit-image` is available.


In [ ]:
def colour_heterogeneity_map(img_rgb: np.ndarray, mask_bin: np.ndarray, use_lab: bool = True) -> np.ndarray:
    mask = mask_bin.astype(bool)
    if mask.sum() == 0:
        return np.zeros(mask_bin.shape, dtype=np.float32)

    if use_lab and SKIMAGE_AVAILABLE:
        colour = rgb2lab(img_rgb).astype(np.float32)
    else:
        colour = img_rgb.astype(np.float32) / 255.0

    lesion_pixels = colour[mask]

    # Median is more robust to extreme pigmentation / highlights than mean.
    typical_colour = np.median(lesion_pixels, axis=0)
    deviation = np.linalg.norm(colour - typical_colour[None, None, :], axis=-1)
    deviation[~mask] = 0.0
    return normalize_01(deviation, mask=mask)


## 8. Combined pseudo-concept generation function

This is the main function to reuse later for XAI evaluation.


In [ ]:
def generate_pseudo_concepts(
    img_rgb_224: np.ndarray,
    mask_bin_224: np.ndarray,
    include_colour: bool = True
) -> dict:
    """Generate pseudo-concept maps for a resized image and resized binary mask.

    Inputs must already be 224 × 224.
    """
    mask = mask_bin_224.astype(np.uint8)

    a = asymmetry_maps(mask)
    b = radial_border_irregularity_map(mask, smooth_sigma=5.0)

    result = {
        "mask": mask.astype(np.uint8),

        # A: asymmetry
        "asymmetry": a["asym_combined"].astype(np.float32),
        "asymmetry_major": a["asym_major"].astype(np.float32),
        "asymmetry_minor": a["asym_minor"].astype(np.float32),

        # B: radial border irregularity
        "border_irregularity": b["border_irregularity"].astype(np.float32),
        "boundary": b["boundary"].astype(np.float32),
        "border_band": b["border_band"].astype(np.float32),
        "radial_deviation": b["radial_deviation"].astype(np.float32),
        "radial_distance": b["radial_distance"].astype(np.float32),
        "radial_smooth": b["radial_smooth"].astype(np.float32),

        # geometry
        "centroid": a["centroid"].astype(np.float32),
        "major_axis": a["major_axis"].astype(np.float32),
        "minor_axis": a["minor_axis"].astype(np.float32),
    }

    if include_colour:
        result["colour_heterogeneity"] = colour_heterogeneity_map(img_rgb_224, mask)

    return result

## 9. Visualisation utilities

Use these before batch generation. The goal is to check whether the maps are plausible, not to prove clinical validity.


In [ ]:
from matplotlib.pyplot import axes


def show_pseudo_concepts_for_row(
    row: pd.Series,
    img_dir: Path,
    mask_dir: Path,
    include_colour: bool = True,
    size=TARGET_SIZE,
):
    stem = row["stem"]
    img = load_rgb(img_dir / f"{stem}.jpg")
    mask = to_mask_bin(load_mask_gray(mask_dir / f"{stem}.png"))

    img_224 = resize_image_rgb(img, size=size)
    mask_224 = resize_mask_bin(mask, size=size)
    concepts = generate_pseudo_concepts(img_224, mask_224, include_colour=include_colour)

    n_cols = 6 if include_colour else 5
    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))

    axes[0].imshow(img_224)
    axes[0].set_title(f"{stem}\nimage 224")
    axes[0].axis("off")

    axes[1].imshow(overlay_mask(img_224, mask_224))
    axes[1].set_title("lesion mask")
    axes[1].axis("off")

    axes[2].imshow(concepts["asymmetry"], cmap="magma", vmin=0, vmax=1)
    axes[2].set_title("A: asymmetry")
    axes[2].axis("off")

    axes[3].imshow(concepts["border_irregularity"], cmap="magma", vmin=0, vmax=1)
    axes[3].set_title("B: radial border irregularity")
    axes[3].axis("off")

    axes[4].imshow(mask_224, cmap="gray")
    axes[4].contour(mask_224, colors="red", linewidths=0.8)
    axes[4].contour(concepts["border_band"], colors="cyan", linewidths=0.8)
    axes[4].set_title("border check\nred=boundary, cyan=border band")
    axes[4].axis("off")

    if include_colour:
        axes[5].imshow(concepts["colour_heterogeneity"], cmap="magma", vmin=0, vmax=1)
        axes[5].set_title("C proxy: colour")
        axes[5].axis("off")

    plt.tight_layout()
    plt.show()

    return concepts


def show_random_pseudo_concepts(
    df: pd.DataFrame,
    img_dir: Path,
    mask_dir: Path,
    n: int = 3,
    random_state: int = RANDOM_STATE,
    include_colour: bool = True,
):
    sample = df.sample(min(n, len(df)), random_state=random_state)
    outputs = {}
    for _, row in sample.iterrows():
        outputs[row["stem"]] = show_pseudo_concepts_for_row(
            row,
            img_dir=img_dir,
            mask_dir=mask_dir,
            include_colour=include_colour,
        )
    return outputs


## 10. Visual sanity check on HAM10000

Run this cell and inspect whether:

- the asymmetry map highlights unmatched lobes or uneven halves
- the border irregularity map is concentrated near locally uneven border segments
- the colour proxy, if enabled, highlights internal colour deviation rather than background


In [ ]:
ham_debug_outputs = show_random_pseudo_concepts(
    ham_core,
    HAM_IMG_DIR,
    HAM_MASK_DIR,
    n=3,
    random_state=21,
    include_colour=True,
)


## 11. Visual sanity check on ISIC 2018


In [ ]:
isic_debug_outputs = show_random_pseudo_concepts(
    isic_core,
    ISIC_IMG_DIR,
    ISIC_MASK_DIR,
    n=3,
    random_state=22,
    include_colour=True,
)


## 12. Save pseudo-concepts for one dataset

The `.npz` file stores all pseudo-concept maps for each image. This is convenient for later XAI evaluation.

Saved keys include:

- `mask`
- `asymmetry`
- `asymmetry_major`
- `asymmetry_minor`
- `border_irregularity`
- `colour_heterogeneity`, if enabled
- debug maps such as `smooth_mask`, `original_boundary`, `smooth_boundary`


In [ ]:
def save_preview_png(stem: str, img_rgb_224: np.ndarray, mask_224: np.ndarray, concepts: dict, out_path: Path):
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))

    axes[0].imshow(img_rgb_224)
    axes[0].set_title("image")
    axes[0].axis("off")

    axes[1].imshow(overlay_mask(img_rgb_224, mask_224))
    axes[1].set_title("mask")
    axes[1].axis("off")

    axes[2].imshow(concepts["asymmetry"], cmap="magma", vmin=0, vmax=1)
    axes[2].set_title("A: asymmetry")
    axes[2].axis("off")

    axes[3].imshow(concepts["border_irregularity"], cmap="magma", vmin=0, vmax=1)
    axes[3].set_title("B: border irregularity")
    axes[3].axis("off")

    fig.suptitle(stem)
    plt.tight_layout()
    fig.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.close(fig)


def generate_and_save_dataset(
    df: pd.DataFrame,
    img_dir: Path,
    mask_dir: Path,
    dataset_name: str,
    include_colour: bool = True,
    max_images: int | None = None,
    save_png_preview: bool = True,
) -> pd.DataFrame:
    rows = []
    work_df = df.copy()
    if max_images is not None:
        work_df = work_df.head(max_images).copy()

    dataset_npz_dir = PSEUDO_NPZ_DIR / dataset_name
    dataset_png_dir = PSEUDO_PNG_DIR / dataset_name
    dataset_npz_dir.mkdir(parents=True, exist_ok=True)
    dataset_png_dir.mkdir(parents=True, exist_ok=True)

    for i, row in work_df.iterrows():
        stem = row["stem"]
        img_path = img_dir / f"{stem}.jpg"
        mask_path = mask_dir / f"{stem}.png"

        img = load_rgb(img_path)
        mask = to_mask_bin(load_mask_gray(mask_path))
        img_224 = resize_image_rgb(img, size=TARGET_SIZE)
        mask_224 = resize_mask_bin(mask, size=TARGET_SIZE)
        concepts = generate_pseudo_concepts(img_224, mask_224, include_colour=include_colour)

        npz_path = dataset_npz_dir / f"{stem}.npz"
        np.savez_compressed(npz_path, **concepts)

        png_path = dataset_png_dir / f"{stem}.png"
        if save_png_preview:
            save_preview_png(stem, img_224, mask_224, concepts, png_path)

        rows.append({
            "dataset": dataset_name,
            "stem": stem,
            "pseudo_npz_path": str(npz_path),
            "preview_png_path": str(png_path) if save_png_preview else None,
            "mask_pixels_224": int(mask_224.sum()),
            "asymmetry_mean": float(concepts["asymmetry"][mask_224.astype(bool)].mean()) if mask_224.sum() else 0.0,
            "border_irregularity_mean": float(concepts["border_irregularity"][mask_224.astype(bool)].mean()) if mask_224.sum() else 0.0,
            "colour_heterogeneity_mean": float(concepts.get("colour_heterogeneity", np.zeros_like(mask_224))[mask_224.astype(bool)].mean()) if mask_224.sum() else 0.0,
        })

        if (len(rows) % 250) == 0:
            print(f"{dataset_name}: processed {len(rows)} / {len(work_df)}")

    manifest = pd.DataFrame(rows)
    manifest_path = PSEUDO_MANIFEST_DIR / f"{dataset_name}_pseudo_concepts_manifest.csv"
    manifest.to_csv(manifest_path, index=False)
    print(f"Saved manifest: {manifest_path.resolve()}")
    print(f"Rows: {len(manifest)}")
    return manifest


## 13. Test saving on a small subset first

Run this before generating the full dataset.


In [ ]:
ham_test_manifest = generate_and_save_dataset(
    ham_core,
    HAM_IMG_DIR,
    HAM_MASK_DIR,
    dataset_name="ham10000_test20",
    include_colour=True,
    max_images=20,
    save_png_preview=True,
)

display(ham_test_manifest.head())


## 14. Full generation cells

Run these only after the visual checks and small subset checks look acceptable.


In [ ]:
# Uncomment when ready.

# ham_pseudo_manifest = generate_and_save_dataset(
#     ham_core,
#     HAM_IMG_DIR,
#     HAM_MASK_DIR,
#     dataset_name="ham10000",
#     include_colour=True,
#     max_images=None,
#     save_png_preview=False,
# )

# isic_pseudo_manifest = generate_and_save_dataset(
#     isic_core,
#     ISIC_IMG_DIR,
#     ISIC_MASK_DIR,
#     dataset_name="isic2018",
#     include_colour=True,
#     max_images=None,
#     save_png_preview=False,
# )


## 15. Later XAI evaluation placeholder

These functions will be used after Grad-CAM, LIME, or SHAP maps are generated.

For now they are included only to show how the pseudo-concept maps will connect to the project problem.


In [ ]:
def saliency_inside_ratio(saliency: np.ndarray, region: np.ndarray, eps: float = 1e-8) -> float:
    """Fraction of saliency intensity inside a binary or soft concept region."""
    s = np.maximum(saliency.astype(np.float32), 0.0)
    r = (region > 0).astype(np.float32)
    return float((s * r).sum() / (s.sum() + eps))


def binary_iou(a: np.ndarray, b: np.ndarray, eps: float = 1e-8) -> float:
    a_bin = a.astype(bool)
    b_bin = b.astype(bool)
    inter = np.logical_and(a_bin, b_bin).sum()
    union = np.logical_or(a_bin, b_bin).sum()
    return float(inter / (union + eps))


def binary_dice(a: np.ndarray, b: np.ndarray, eps: float = 1e-8) -> float:
    a_bin = a.astype(bool)
    b_bin = b.astype(bool)
    inter = np.logical_and(a_bin, b_bin).sum()
    return float(2 * inter / (a_bin.sum() + b_bin.sum() + eps))


def concept_alignment_summary(saliency: np.ndarray, concepts: dict, threshold: float = 0.5) -> dict:
    """Compute basic saliency alignment metrics for pseudo-concept maps.

    This expects saliency already resized to 224 × 224 and normalised to [0, 1].
    """
    sal_bin = saliency >= threshold
    rows = {}

    for name in ["mask", "asymmetry", "border_irregularity", "colour_heterogeneity"]:
        if name not in concepts:
            continue
        region = concepts[name]
        region_bin = region > 0
        rows[f"{name}_sir"] = saliency_inside_ratio(saliency, region_bin)
        rows[f"{name}_iou"] = binary_iou(sal_bin, region_bin)
        rows[f"{name}_dice"] = binary_dice(sal_bin, region_bin)

    return rows


## 16. Notes for the report

Use cautious wording:

> The pseudo-concept maps do not aim to reproduce clinical ABCD assessment. They provide spatially localised, heuristic approximations of selected diagnostic cues, enabling structured evaluation of model explanations beyond lesion-level overlap.

Recommended final scope:

- Main: A and B
- Optional: C as a weak image-derived proxy
- Excluded: D, because physical scale is unavailable
